In [42]:
# US CENSUS DATA CLEANING PIPELINE
# Clean and Prepare Dataset for PostgreSQL Database Upload
# Install Required Packages
# pip install pandas numpy sqlalchemy psycopg2-binary

import pandas as pd
import numpy as np
import re
from sqlalchemy import create_engine

# STEP 1: LOAD DATASET

# Load CSV exported from Census API extraction

df = pd.read_csv("us_census_county_data.csv")

print("Original Dataset Shape:", df.shape)

# STEP 2: STANDARDIZE COLUMN NAMES

# Convert column names to lowercase
# Replace spaces with underscores
# Remove special characters

df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)

print("\nStandardized Column Names:")
print(df.columns)

Original Dataset Shape: (3222, 14)

Standardized Column Names:
Index(['countystate_name', 'population', 'median_household_income',
       'people_below_poverty', 'employed_population', 'unemployed_population',
       'bachelors_degree', 'median_home_value', 'state', 'county',
       'poverty_rate', 'employment_rate', 'unemployment_rate',
       'education_rate'],
      dtype='str')


In [43]:
# STEP 3: REMOVE DUPLICATES

# Remove complete duplicate rows

duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print(f"\nDuplicates Removed: {duplicates_before - duplicates_after}")

# Remove duplicates based on unique geographic identifiers

if {'state', 'county'}.issubset(df.columns):
    df = df.drop_duplicates(subset=['state', 'county'])

# STEP 4: HANDLE MISSING VALUES

# Replace common missing value indicators

missing_values = [
    "", " ", "NA", "N/A", "NULL",
    "null", "-", "--", "nan"
]

df.replace(missing_values, np.nan, inplace=True)

# STEP 5: CLEAN TEXT FIELDS

# Identify text columns

text_columns = df.select_dtypes(include='object').columns

for col in text_columns:

    # Convert to string
    df[col] = df[col].astype(str)

    # Remove leading/trailing spaces
    df[col] = df[col].str.strip()

    # Remove multiple spaces
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

    # Remove unwanted special characters
    df[col] = df[col].str.replace(
        r"[^a-zA-Z0-9\s,.\-()/]",
        "",
        regex=True
    )

    # Convert empty strings to NaN
    df[col] = df[col].replace("", np.nan)


# STEP 6: HANDLE MISSING TEXT VALUES

for col in text_columns:

    df[col] = df[col].fillna("Unknown")


Duplicates Removed: 0


C:\Users\Patrick\AppData\Local\Temp\ipykernel_13932\2924198309.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include='object').columns


In [44]:
# STEP 7: CLEAN NUMERIC COLUMNS

# Identify potential numeric columns

numeric_columns = [
    'population',
    'median_household_income',
    'people_below_poverty',
    'employed_population',
    'unemployed_population',
    'bachelors_degree',
    'median_home_value'
]

# Convert numeric columns

for col in numeric_columns:

    if col in df.columns:

        # Remove commas and dollar signs
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .str.replace("$", "")
        )

        # Convert to numeric
        df[col] = pd.to_numeric(df[col], errors='coerce')


# STEP 8: HANDLE MISSING NUMERIC VALUES

for col in numeric_columns:

    if col in df.columns:

        # Replace missing values with median
        median_value = df[col].median()

        df[col] = df[col].fillna(median_value)

In [45]:
# STEP 9: REMOVE INVALID OR NEGATIVE VALUES

# Remove negative values from population/economic fields

for col in numeric_columns:

    if col in df.columns:

        df[col] = np.where(df[col] < 0, np.nan, df[col])

In [46]:
# STEP 10: CREATE ANALYTICAL COLUMNS

# Poverty Rate

if {
    'people_below_poverty',
    'population'
}.issubset(df.columns):

    df['poverty_rate'] = (
        df['people_below_poverty'] /
        df['population']
    ) * 100

# Employment Rate

if {
    'employed_population',
    'unemployed_population'
}.issubset(df.columns):

    labor_force = (
        df['employed_population'] +
        df['unemployed_population']
    )

    df['employment_rate'] = (
        df['employed_population'] /
        labor_force
    ) * 100

# Unemployment Rate

if {
    'employed_population',
    'unemployed_population'
}.issubset(df.columns):

    labor_force = (
        df['employed_population'] +
        df['unemployed_population']
    )

    df['unemployment_rate'] = (
        df['unemployed_population'] /
        labor_force
    ) * 100

# Education Rate

if {
    'bachelors_degree',
    'population'
}.issubset(df.columns):

    df['education_rate'] = (
        df['bachelors_degree'] /
        df['population']
    ) * 100

In [47]:
# STEP 11: FORMAT DATA TYPES

# Convert geographic codes to string

geo_columns = ['state', 'county']

for col in geo_columns:

    if col in df.columns:

        df[col] = df[col].astype(str)

In [48]:
# STEP 12: REMOVE ROWS WITH EXCESSIVE MISSING DATA

# Remove rows with more than 50% missing values

threshold = len(df.columns) * 0.5

df = df.dropna(thresh=threshold)

In [49]:
# STEP 13: VALIDATE DATASET

print("\nCleaned Dataset Information:")
print(df.info())

print("\nMissing Values Summary:")
print(df.isnull().sum())

print("\nDataset Shape After Cleaning:")
print(df.shape)

# STEP 14: EXPORT CLEANED DATASET

df.to_csv("cleaned_us_census_data.csv", index=False)

print("\nCleaned CSV file saved successfully.")


Cleaned Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 3222 entries, 0 to 3221
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   countystate_name         3222 non-null   str    
 1   population               3222 non-null   float64
 2   median_household_income  3220 non-null   float64
 3   people_below_poverty     3222 non-null   float64
 4   employed_population      3222 non-null   float64
 5   unemployed_population    3222 non-null   float64
 6   bachelors_degree         3222 non-null   float64
 7   median_home_value        3216 non-null   float64
 8   state                    3222 non-null   str    
 9   county                   3222 non-null   str    
 10  poverty_rate             3222 non-null   float64
 11  employment_rate          3222 non-null   float64
 12  unemployment_rate        3222 non-null   float64
 13  education_rate           3222 non-null   float64
dtypes: fl